# 71 — Profile Agent 70 (deferred ships_sent filter)
Runs `70-Polars_filter.py` (player 0) vs a random agent (player 1) for up to 100 steps.
Measures wall-clock time per step and uses `line_profiler` to show per-line hotspots.

**Optimisation vs Agent 68:** ships_sent expansion is deferred until after a planet-level
MAX_SPEED reachability filter. The cross-join is now N_src_planets × N_targets × N_steps
(no ships_sent dimension), reducing it by the average ships_sent range (50–200×).
`planet_disp_lf` provides actual per-tick displacement from simulation data for all
nature types (fix ≈ 0, moving = orbital arc, comet = path step).

In [ ]:
import importlib.util
import time
import random
import math

import kaggle_environments as ke
import plotly.graph_objects as go
from line_profiler import LineProfiler

In [ ]:
# Load 70-Polars_filter.py as a fresh module — resets global step/player state
spec = importlib.util.spec_from_file_location("agent70", "70-Polars_filter.py")
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
m.step = 0
m.num_agents = None
m.player_id = None

In [ ]:
SEED = 42
N_STEPS = 100
random.seed(SEED)

def random_agent_fn(obs):
    player = obs.player
    my_planets = [p for p in obs.planets if p[1] == player]
    if not my_planets:
        return []
    planet = random.choice(my_planets)
    ships = planet[5] // 2
    if ships < 1:
        return []
    return [[planet[0], random.uniform(0, 2 * math.pi), ships]]

In [ ]:
# Instrument key functions with line_profiler
lp = LineProfiler()
lp.add_function(m.swept_pair_hit)
lp.add_function(m._simulate)
lp.add_function(m.take_action)
profiled_agent = lp(m.nearest_planet_sniper)

In [ ]:
env = ke.make("orbit_wars", debug=False)
env.reset(2)

step_numbers: list[int] = []
step_times_ms: list[float] = []

for env_step in range(N_STEPS):
    obs0 = env.state[0].observation
    obs1 = env.state[1].observation

    t0 = time.perf_counter()
    action0 = profiled_agent(obs0)
    dt_ms = (time.perf_counter() - t0) * 1000
    step_numbers.append(env_step)
    step_times_ms.append(dt_ms)
    print(f"Step {env_step:3d}: {dt_ms:7.2f} ms")

    action1 = random_agent_fn(obs1)
    env.step([action0, action1])

In [ ]:
obs0 = env.state[0].observation
p0 = sum(p[5] for p in obs0.planets if p[1] == 0)
p1 = sum(p[5] for p in obs0.planets if p[1] == 1)
n = len(step_numbers)
mean_ms = sum(step_times_ms) / n

print(f"Player 0 (our agent): {p0} ships")
print(f"Player 1 (random):    {p1} ships")
winner = "Our agent wins" if p0 > p1 else "Random wins" if p1 > p0 else "Tie"
print(f"Result after {n} steps: {winner}")
print(f"Step times — min: {min(step_times_ms):.1f} ms  max: {max(step_times_ms):.1f} ms  mean: {mean_ms:.1f} ms")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=step_numbers,
    y=step_times_ms,
    mode="lines+markers",
    name="Agent call time",
    line=dict(color="dodgerblue", width=1.5),
    marker=dict(size=5),
))
fig.add_hline(
    y=mean_ms,
    line_dash="dash",
    line_color="orange",
    annotation_text=f"Mean {mean_ms:.1f} ms",
    annotation_position="top right",
)
fig.update_layout(
    title="70-Polars_filter agent call time per step (vs random)",
    xaxis_title="Step",
    yaxis_title="Time (ms)",
    template="plotly_dark",
)
fig.show()

In [ ]:
import io

buf = io.StringIO()
lp.print_stats(stream=buf, output_unit=1e-3)
print(buf.getvalue())